<a href="https://colab.research.google.com/github/TylerWichman/Tyler_Wichman_Portfolio/blob/main/Fantasy_Trade_Calculator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Fantasy Trade Calculator

A dynasty/redraft fantasy football trade calculator that trains supervised gradient-boosting
models to predict market consensus value (FantasyCalc, blended with KeepTradeCut) from player
fundamentals, producing one unified value board per format that puts every player and draft pick
on the same directly-comparable scale.

**Consensus value is the training target, not a benchmark to beat.** These models are trained to
reflect the fantasy trade market as closely as possible -- agreement with FantasyCalc/KeepTradeCut
consensus is the design goal, not an independent finding. Where a player's model-predicted value
diverges from their current market price, that divergence is reported as a residual: a candidate
worth investigating, never a claim that the model is right and the market is wrong.

Full methodology, scope, and known exclusions: see `README.md` in this directory.

This notebook is a thin orchestration layer -- all real logic lives in
`trade_calculator_pipeline.py`, which is imported and called stage by stage below.

In [ ]:
import pandas as pd

import trade_calculator_pipeline as tcp

pd.set_option("display.max_rows", 250)
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)

## Stage 1 -- ID crosswalk

Built from nflreadpy's `load_ff_playerids()`, the backbone every source join uses.

In [ ]:
crosswalk = tcp.build_crosswalk()
print(f"crosswalk rows: {len(crosswalk)}")
crosswalk.head()

crosswalk rows: 12472


,mfl_id,sleeper_id,espn_id,gsis_id,ktc_id,pfr_id,name,position,merge_name
0,17462,13269.0,4837248.0,00-0041562,1924.0,MendFe00,Fernando Mendoza,QB,fernando mendoza
1,17463,13275.0,4685522.0,00-0041568,1925.0,SimpTy00,Ty Simpson,QB,ty simpson
2,17464,NaN,NaN,NaN,NaN,NaN,Trinidad Chambliss,XX,trinidad chambliss
3,17465,13404.0,4567747.0,00-0040906,1927.0,NussGa00,Garrett Nussmeier,QB,garrett nussmeier
4,17466,13272.0,4430841.0,00-0041561,1928.0,BeckCa01,Carson Beck,QB,carson beck


## Stage 2 -- Pull FantasyCalc (6 format-teamsize combos) and KeepTradeCut (dynasty + redraft)

In [ ]:
fc_all = tcp.pull_all_fantasycalc()
for k, v in fc_all.items():
    print(f"  {k}: {len(v)} rows")

ktc_dyn = tcp.pull_ktc_dynasty()
ktc_redraft = tcp.pull_ktc_redraft()
print(f"ktc_dynasty: {len(ktc_dyn)} rows, ktc_redraft: {len(ktc_redraft)} rows")

  ('sf_dynasty', 10): 475 rows
  ('sf_dynasty', 12): 475 rows
  ('oneqb_dynasty', 10): 475 rows
  ('oneqb_dynasty', 12): 475 rows
  ('redraft', 10): 171 rows
  ('redraft', 12): 171 rows


ktc_dynasty: 500 rows, ktc_redraft: 300 rows


### Blend example -- SF dynasty, 12 team

A quick look at one format's crosswalk join + percentile blend before running the full 6-combo
version in Stage 7.

In [ ]:
fc_joined_demo = tcp.join_fantasycalc_crosswalk(fc_all[("sf_dynasty", 12)], crosswalk, label="FC-SF12")
ktc_joined_demo = tcp.join_ktc_crosswalk(ktc_dyn, crosswalk, label="KTC-dyn")
blended_demo = tcp.blend_consensus(fc_joined_demo, ktc_joined_demo, ktc_value_col="sf_value")
blended_demo[["name", "position", "value", "ktc_percentile", "fc_percentile", "consensus_value"]] \
    .sort_values("consensus_value", ascending=False).head(15)

[FC-SF12] 2 rows matched via normalized-name fallback (documented last resort)
[FC-SF12] top-200 join coverage: 100.0%
[KTC-dyn] 7 unmatched after fallback: ['Hassan Haskins', 'Stetson Bennett', 'Dylan Laube', 'Thomas Fidone', 'Sterling Shepard', 'Foster Moreau', 'Tim Patrick']
[KTC-dyn] top-200 join coverage: 100.0%


,name,position,value,ktc_percentile,fc_percentile,consensus_value
0,Bijan Robinson,RB,10438,99.781182,100.000000,10373.074617
2,Jahmyr Gibbs,RB,10189,100.000000,99.498747,10214.317293
1,Josh Allen,QB,10225,99.343545,99.749373,10204.738734
3,Ja'Marr Chase,WR,10080,99.562363,99.248120,10128.533254
4,Jaxon Smith-Njigba,WR,8777,99.124726,98.997494,9021.000136
5,Drake Maye,QB,8595,98.905908,98.746867,8637.601712
6,Puka Nacua,WR,8308,98.687090,98.496241,8388.615548
7,Brock Bowers,TE,7606,98.468271,98.245614,7836.049246
9,Amon-Ra St. Brown,WR,7264,98.249453,97.744361,7287.962937
8,Lamar Jackson,QB,7297,97.592998,97.994987,7279.182592


## Stage 3 -- Feature assembly

Draft capital, season-level production/efficiency, continuous position-specific aging curves, and
combine metrics (RB speed score).

In [ ]:
draft_capital = tcp.load_draft_capital(crosswalk)
print(f"draft_capital rows: {len(draft_capital)}")

weekly = tcp.load_weekly_features()
print(f"weekly/season feature rows: {len(weekly)}, cols: {len(weekly.columns)}")

curve_params, peak_ages, aged = tcp.fit_aging_curves(weekly, draft_capital)
print(f"peak ages by position: { {k: round(v, 1) for k, v in peak_ages.items()} }")

combine_feats = tcp.load_combine_features()
print(f"combine feature rows: {len(combine_feats)}")
combine_feats.dropna(subset=["speed_score"]).head(3)

[draft_capital] 2 duplicate gsis_id rows in load_draft_picks -- keeping first (data-entry dupes, e.g. supplemental draft)
[draft_capital] 73 rows had a non-gsis id from load_draft_picks (new/unplayed draft class) -- 73/73 repaired via crosswalk name match
[draft_capital] 1 rows still unresolved after repair attempt: ['Mike Washington Jr.']
draft_capital rows: 3551


[snap_share] 1.0% of player-season snap rows unmatched to gsis_id via pfr_id (dropped)
weekly/season feature rows: 4191, cols: 27
[aging_curve] 1261 / 4191 player-seasons have no draft-capital age match (undrafted players) -- age kept as NaN, excluded only from curve fitting
peak ages by position: {'QB': 27.0, 'RB': np.float64(26.0), 'WR': np.float64(27.2), 'TE': np.float64(32.2)}


[combine] gsis_id coverage: 64.9% (pfr_id only) -> 70.2% after normalized-name fallback (160 rows recovered)
combine feature rows: 2103


,gsis_id,forty_time,combine_weight,combine_height,speed_score
56,00-0019641,4.45,216.0,5-10,110.165016
125,00-0020270,4.53,226.0,6-0,107.336054
118,00-0020514,4.38,207.0,5-9,112.487405


### Rookie college-production features (CollegeFootballData)

Dominator rating (80% yardage / 20% TD weighted share of team receiving output) and breakout age
for the current draft class. **Note:** CFBD's free-tier monthly call quota can be exhausted at
build time -- if so, this degrades gracefully to `NaN` for affected rookies, who then fall back to
draft capital + age + combine speed score alone (`HistGradientBoostingRegressor` handles missing
features natively; see README's "Known exclusions").

In [ ]:
rookie_college = tcp.compute_rookie_college_features(draft_capital, draft_year=2026)
coverage = (rookie_college["college_seasons_found"] > 0).mean()
print(f"2026 rookie CFBD college-production coverage: {coverage:.1%}"
      + ("  (degraded -- CFBD quota likely exhausted this cycle)" if coverage < 0.3 else ""))
rookie_college.sort_values("dominator_rating", ascending=False).head(10)

[cfbd_rookie] 0.0% of 2026 skill draftees matched to >=1 CFBD college season with usable receiving-dominator data (QBs excluded from this stat by design)
[cfbd_rookie] failure breakdown: 73/73 hit CFBD's monthly call quota (expected, documented free-tier limit -- resets monthly on CFBD's own cycle, re-run once available; low coverage from this alone is NOT a bug), 0/73 failed for a DIFFERENT reason (network error, non-quota HTTP error, or a parsing bug -- investigate this count specifically if it's nonzero, it means something other than the quota is the problem)
2026 rookie CFBD college-production coverage: 0.0%  (degraded -- CFBD quota likely exhausted this cycle)


,gsis_id,dominator_rating,breakout_age,college_seasons_found
0,00-0041562,NaN,NaN,0
1,00-0041027,NaN,NaN,0
2,00-0041438,NaN,NaN,0
3,00-0041029,NaN,NaN,0
4,00-0041568,NaN,NaN,0
5,00-0041032,NaN,NaN,0
6,00-0040867,NaN,NaN,0
7,00-0041547,NaN,NaN,0
8,00-0041511,NaN,NaN,0
9,00-0041512,NaN,NaN,0


## Stage 4 -- Draft pick valuation

2027-2029 x rounds 1-3 x {Early, Mid, Late} tiers, priced from native FantasyCalc/KeepTradeCut
market data where available, constructed via a year-discount/class-strength/format-multiplier
formula elsewhere. Redraft formats have no pick assets.

In [ ]:
pick_universe = tcp.build_pick_universe(fc_all, ktc_dyn)
print(f"pick_universe rows: {len(pick_universe)}")
pick_universe[(pick_universe["format"] == "sf_dynasty") & (pick_universe["team_size"] == 12)] \
    .sort_values("pick_value", ascending=False)[["year", "round", "tier", "fc_value", "pick_value"]]

[format_multiplier] overall mean SF/1QB pick-value ratio: 1.060
[format_multiplier] by round: {1: 1.061, 2: 1.061, 3: 1.06, 4: 1.059}
[format_multiplier] NOTE: spec §6 cites a 30-60% player-level SF premium for elite QBs; picks came out much lower (~6%). That's expected, not a bug -- a pick is position-agnostic (might become a WR/RB/etc.), so it can't carry a specific elite-QB premium the way a known franchise QB player does.


[pick_universe] 36 native FC tier prices, 72 FC-derived (aggregate x tier-ratio), 0 fully constructed (of 108 total)
pick_universe rows: 108


,year,round,tier,fc_value,pick_value
27,2027,1,Early,4532.000000,5388.800000
36,2028,1,Early,3190.750270,3886.987676
28,2027,1,Mid,2984.000000,3863.900000
29,2027,1,Late,2297.000000,3232.550000
37,2028,1,Mid,2101.090720,2963.808968
45,2029,1,Early,2910.832307,2910.832307
30,2027,2,Early,1843.000000,2495.400000
38,2028,1,Late,1617.233964,2475.002076
31,2027,2,Mid,1528.000000,2180.400000
39,2028,2,Early,1583.333802,2143.216971


## Stage 5 -- Unified feature table

One row per player, scoped to the *active* FantasyCalc asset universe (spec §2a) -- not however far back the historical production pull happens to reach. Earlier runs let long-retired players leak into the intermediate feature table (ages 44-49 observed); this version filters to the actual set of gsis_ids FantasyCalc prices across all 6 pulls before anything downstream touches it.

In [ ]:
active_ids = tcp.active_asset_gsis_ids(fc_all, crosswalk)
print(f"active FantasyCalc asset universe: {len(active_ids)} gsis_ids")
snapshot = tcp.build_player_snapshot(aged, draft_capital, peak_ages, active_ids=active_ids)
feature_table = tcp.build_feature_table(snapshot, draft_capital, combine_feats, rookie_college)
feature_table[["gsis_id", "position", "current_age", "career_games", "ppg_ppr"]].head()

[active-universe-sf_dynasty10] 2 rows matched via normalized-name fallback (documented last resort)
[active-universe-sf_dynasty10] top-200 join coverage: 100.0%
[active-universe-sf_dynasty12] 2 rows matched via normalized-name fallback (documented last resort)
[active-universe-sf_dynasty12] top-200 join coverage: 100.0%


[active-universe-oneqb_dynasty10] 2 rows matched via normalized-name fallback (documented last resort)
[active-universe-oneqb_dynasty10] top-200 join coverage: 100.0%


[active-universe-oneqb_dynasty12] 2 rows matched via normalized-name fallback (documented last resort)
[active-universe-oneqb_dynasty12] top-200 join coverage: 100.0%
[active-universe-redraft10] top-200 join coverage: 100.0%


[active-universe-redraft12] top-200 join coverage: 100.0%
active FantasyCalc asset universe: 399 gsis_ids


[snapshot] WARNING: 17 of 73 2026 rookies are NOT in the active FantasyCalc universe -- unexpected, since FantasyCalc prices rookies too; they will be dropped along with everyone else outside the active universe
[snapshot] scoped to active FantasyCalc asset universe: 1422 -> 380 rows
[snapshot] 380 player rows (73 zero-snap 2026 rookies appended with NaN production/career features)
[feature_table] 380 rows, 44 columns


,gsis_id,position,current_age,career_games,ppg_ppr
0,00-0023459,QB,42.0,99.0,14.192500
1,00-0026158,QB,41.0,45.0,11.281538
2,00-0026498,QB,38.0,98.0,20.610588
3,00-0028118,QB,37.0,32.0,9.910000
4,00-0029604,QB,38.0,96.0,10.354000


## Stage 6 -- Consensus targets for all 6 format-teamsize combos

In [ ]:
consensus_targets = tcp.build_all_consensus_targets(fc_all, ktc_dyn, ktc_redraft, crosswalk)
for k, v in consensus_targets.items():
    print(f"  {k}: {len(v)} players with a consensus target")

[KTC-dyn] 7 unmatched after fallback: ['Hassan Haskins', 'Stetson Bennett', 'Dylan Laube', 'Thomas Fidone', 'Sterling Shepard', 'Foster Moreau', 'Tim Patrick']
[KTC-dyn] top-200 join coverage: 100.0%
[ktc_redraft_bridge] 281/300 rows bridged to dynasty-page playerID by name (93.7%)
[KTC-redraft] 14 unmatched after fallback: ['Joe Mixon', 'Nick Chubb', 'Miles Sanders', 'DeAndre Hopkins', 'Adam Thielen', 'Jermaine Burton', 'Tyler Lockett', 'Hunter Renfrow', 'Austin Ekeler', 'Brandin Cooks', 'Zay Jones', 'Taysom Hill', 'Russell Wilson', 'Raheem Mostert']
[KTC-redraft] top-200 join coverage: 99.5%


[FC-sf_dynasty10] 2 rows matched via normalized-name fallback (documented last resort)
[FC-sf_dynasty10] top-200 join coverage: 100.0%


[FC-sf_dynasty12] 2 rows matched via normalized-name fallback (documented last resort)
[FC-sf_dynasty12] top-200 join coverage: 100.0%
[FC-oneqb_dynasty10] 2 rows matched via normalized-name fallback (documented last resort)
[FC-oneqb_dynasty10] top-200 join coverage: 100.0%


[FC-oneqb_dynasty12] 2 rows matched via normalized-name fallback (documented last resort)


[FC-oneqb_dynasty12] top-200 join coverage: 100.0%
[FC-redraft10] top-200 join coverage: 100.0%


[FC-redraft12] top-200 join coverage: 100.0%
  ('sf_dynasty', 10): 399 players with a consensus target
  ('sf_dynasty', 12): 399 players with a consensus target
  ('oneqb_dynasty', 10): 399 players with a consensus target
  ('oneqb_dynasty', 12): 399 players with a consensus target
  ('redraft', 10): 171 players with a consensus target
  ('redraft', 12): 171 players with a consensus target


## Stage 7 -- Fit all 6 models

One `HistGradientBoostingRegressor` per format-teamsize combo, trained on `log(consensus_value)`
with `position` as a native categorical feature, validated via 10-fold out-of-fold cross-validated
predictions (not train-set fit metrics). A `RandomForestRegressor` (imputed/encoded) serves as a
comparison baseline. Zero-snap rookies get an auxiliary lower-quantile prediction as their final
value instead of the mean -- a genuine predictive-uncertainty discount, never a neutral risk
multiplier.

**Sample-weighted training.** `HistGradientBoostingRegressor`'s default `min_samples_leaf=20`,
combined with this project's ~379-row training pool, structurally forces the top ~15-20 "elite tier"
players -- in any position, any format -- into leaves blended with lower-value neighbors, since every
leaf must average at least 20 samples. That systematically compresses predictions for the most
valuable assets (first surfaced as superflex elite-QB underprediction -- Josh Allen, Lamar Jackson,
Jayden Daniels were predicted well below their own training target -- then confirmed as a general,
position-agnostic effect across the whole board). The fix: every training row is weighted by its own
percentile rank in that format's `consensus_value` distribution (`weight = 1 + percentile**3`,
purely a function of where a row sits in its own format's target -- no position or format is
special-cased anywhere), combined with a lower `min_samples_leaf` (10). Verified across all 6 formats
with a multi-seed robustness check: this fixes the elite-QB compression as well as an earlier
QB-only board-assembly blend did, *and* also fixes the smaller RB/WR/TE top-of-board compression that
blend never touched, with Spearman/MAE flat-to-improved everywhere -- not a tradeoff against overall
accuracy. See `trade_calculator_pipeline.py`'s module comment above `fit_format_model` for the full
diagnosis, including the two earlier fixes (a QB-only model split, a starter-status feature) that
were tried and honestly ruled out first.

In [ ]:
fit_results = tcp.fit_all_format_models(feature_table, consensus_targets)

validation_rows = []
for (fmt, team_size), res in fit_results.items():
    hm, rm = res["hgb_metrics"], res["rf_metrics"]
    validation_rows.append({
        "format": fmt, "team_size": team_size,
        "hgb_spearman": hm["spearman"], "hgb_mae": hm["mae"], "hgb_top50_mae": hm["top50_mae"],
        "rf_spearman": rm["spearman"], "rf_mae": rm["mae"],
    })
validation_df = pd.DataFrame(validation_rows)
validation_df

[sf_dynasty-10] dropping 2 feature(s) with <2 distinct values in this group: ['breakout_age', 'dominator_rating']


[sf_dynasty-10] rookie quantile discount: 56 rookies in training pool, mean-model vs quantile-model prediction reduced by 2.5% on average
[sf_dynasty-10] n=380  HGBR: spearman=0.824 mae=660 top50_mae=1605  |  RF baseline: spearman=0.833 mae=631 top50_mae=1730
[sf_dynasty-12] dropping 2 feature(s) with <2 distinct values in this group: ['breakout_age', 'dominator_rating']


[sf_dynasty-12] rookie quantile discount: 56 rookies in training pool, mean-model vs quantile-model prediction reduced by 3.1% on average
[sf_dynasty-12] n=380  HGBR: spearman=0.823 mae=658 top50_mae=1590  |  RF baseline: spearman=0.831 mae=639 top50_mae=1755
[oneqb_dynasty-10] dropping 2 feature(s) with <2 distinct values in this group: ['breakout_age', 'dominator_rating']


[oneqb_dynasty-10] rookie quantile discount: 56 rookies in training pool, mean-model vs quantile-model prediction reduced by 4.6% on average
[oneqb_dynasty-10] n=380  HGBR: spearman=0.836 mae=544 top50_mae=1314  |  RF baseline: spearman=0.835 mae=577 top50_mae=1709
[oneqb_dynasty-12] dropping 2 feature(s) with <2 distinct values in this group: ['breakout_age', 'dominator_rating']


[oneqb_dynasty-12] rookie quantile discount: 56 rookies in training pool, mean-model vs quantile-model prediction reduced by 6.2% on average
[oneqb_dynasty-12] n=380  HGBR: spearman=0.828 mae=557 top50_mae=1370  |  RF baseline: spearman=0.834 mae=575 top50_mae=1709
[redraft-10] dropping 2 feature(s) with <2 distinct values in this group: ['breakout_age', 'dominator_rating']


[redraft-10] rookie quantile discount: 13 rookies in training pool, mean-model vs quantile-model prediction reduced by 11.4% on average
[redraft-10] n=169  HGBR: spearman=0.751 mae=915 top50_mae=1712  |  RF baseline: spearman=0.763 mae=991 top50_mae=1983
[redraft-12] dropping 2 feature(s) with <2 distinct values in this group: ['breakout_age', 'dominator_rating']


[redraft-12] rookie quantile discount: 13 rookies in training pool, mean-model vs quantile-model prediction reduced by 14.6% on average
[redraft-12] n=169  HGBR: spearman=0.752 mae=908 top50_mae=1656  |  RF baseline: spearman=0.760 mae=994 top50_mae=1957


,format,team_size,hgb_spearman,hgb_mae,hgb_top50_mae,rf_spearman,rf_mae
0,sf_dynasty,10,0.823642,660.146610,1604.945051,0.832779,630.737997
1,sf_dynasty,12,0.822548,657.691022,1589.671815,0.831399,638.884803
2,oneqb_dynasty,10,0.836388,544.228060,1313.753733,0.835340,576.505584
3,oneqb_dynasty,12,0.828104,557.285482,1370.222670,0.833904,575.476279
4,redraft,10,0.750902,915.111652,1711.978995,0.763182,990.620102
5,redraft,12,0.751875,908.487251,1656.231438,0.760382,993.990372


## Stage 8 -- Board assembly

Every player the model produced a final prediction for, plus (dynasty formats only) the 27 pick
assets, all on the same `model_value` scale -- the trade-calculator itself: any two assets, player
or pick, are now directly comparable.

In [ ]:
boards = tcp.build_all_boards(feature_table, crosswalk, consensus_targets, pick_universe, fit_results)
for (fmt, team_size), board in boards.items():
    n_players = (board["asset_type"] == "player").sum()
    n_picks = (board["asset_type"] == "pick").sum()
    print(f"  {fmt}-{team_size}: {len(board)} rows ({n_players} players, {n_picks} picks)")

  sf_dynasty-10: 407 rows (380 players, 27 picks)
  sf_dynasty-12: 407 rows (380 players, 27 picks)
  oneqb_dynasty-10: 407 rows (380 players, 27 picks)
  oneqb_dynasty-12: 407 rows (380 players, 27 picks)
  redraft-10: 169 rows (169 players, 0 picks)
  redraft-12: 169 rows (169 players, 0 picks)


### Top 15, SF dynasty (12-team) -- eye-test sanity check

Should be recognizable star players, maybe a high pick mixed in near the very top -- **not** a
bottom-tier pick outranking Bijan Robinson.

In [ ]:
boards[("sf_dynasty", 12)][["board_rank", "name", "position", "asset_type", "model_value", "consensus_value"]].head(15)

,board_rank,name,position,asset_type,model_value,consensus_value
0,1,Jahmyr Gibbs,RB,player,10360.832614,10214.317293
1,2,Bijan Robinson,RB,player,10167.173204,10373.074617
2,3,Ja'Marr Chase,WR,player,10166.628076,10128.533254
3,4,Josh Allen,QB,player,9686.623359,10204.738734
4,5,Jaxon Smith-Njigba,WR,player,8884.012524,9021.000136
5,6,Drake Maye,QB,player,8726.869730,8637.601712
6,7,Puka Nacua,WR,player,8340.308640,8388.615548
7,8,Brock Bowers,TE,player,7872.213374,7836.049246
8,9,Amon-Ra St. Brown,WR,player,7371.649480,7287.962937
9,10,Caleb Williams,QB,player,7299.233430,7245.616469


## Top 200 -- every board

The full trade-calculator board for each of the 6 format-teamsize combos.

### Superflex Dynasty -- 10 team -- Top 200

In [ ]:
boards[("sf_dynasty", 10)][
    ["board_rank", "name", "position", "asset_type", "model_value", "consensus_value", "current_age"]
].head(200)

,board_rank,name,position,asset_type,model_value,consensus_value,current_age
0,1,Jahmyr Gibbs,RB,player,10516.887564,10385.258897,24.0
1,2,Bijan Robinson,RB,player,10243.481059,10471.491685,24.0
2,3,Ja'Marr Chase,WR,player,10003.743141,10047.748288,26.0
3,4,Josh Allen,QB,player,9769.384532,9915.415205,30.0
4,5,Jaxon Smith-Njigba,WR,player,8737.508926,8945.469040,24.0
5,6,Drake Maye,QB,player,8423.947820,8415.034589,24.0
6,7,Puka Nacua,WR,player,8249.336056,8276.853425,25.0
7,8,Brock Bowers,TE,player,7847.238734,7790.738424,23.0
8,9,Ashton Jeanty,RB,player,7280.301733,7296.959469,22.0
9,10,Amon-Ra St. Brown,WR,player,7269.614423,7293.162663,26.0


### Superflex Dynasty -- 12 team -- Top 200

In [ ]:
boards[("sf_dynasty", 12)][
    ["board_rank", "name", "position", "asset_type", "model_value", "consensus_value", "current_age"]
].head(200)

,board_rank,name,position,asset_type,model_value,consensus_value,current_age
0,1,Jahmyr Gibbs,RB,player,10360.832614,10214.317293,24.0
1,2,Bijan Robinson,RB,player,10167.173204,10373.074617,24.0
2,3,Ja'Marr Chase,WR,player,10166.628076,10128.533254,26.0
3,4,Josh Allen,QB,player,9686.623359,10204.738734,30.0
4,5,Jaxon Smith-Njigba,WR,player,8884.012524,9021.000136,24.0
5,6,Drake Maye,QB,player,8726.869730,8637.601712,24.0
6,7,Puka Nacua,WR,player,8340.308640,8388.615548,25.0
7,8,Brock Bowers,TE,player,7872.213374,7836.049246,23.0
8,9,Amon-Ra St. Brown,WR,player,7371.649480,7287.962937,26.0
9,10,Caleb Williams,QB,player,7299.233430,7245.616469,24.0


### 1QB Dynasty -- 10 team -- Top 200

In [ ]:
boards[("oneqb_dynasty", 10)][
    ["board_rank", "name", "position", "asset_type", "model_value", "consensus_value", "current_age"]
].head(200)

,board_rank,name,position,asset_type,model_value,consensus_value,current_age
0,1,Jahmyr Gibbs,RB,player,11218.069399,11127.789959,24.0
1,2,Bijan Robinson,RB,player,11098.299781,11302.850109,24.0
2,3,Ja'Marr Chase,WR,player,10144.295302,9993.607123,26.0
3,4,Jaxon Smith-Njigba,WR,player,8716.428940,8783.488972,24.0
4,5,Puka Nacua,WR,player,8258.666404,8230.139726,25.0
5,6,Ashton Jeanty,RB,player,7727.755146,7675.337060,22.0
6,7,Amon-Ra St. Brown,WR,player,7293.373581,7265.846961,26.0
7,8,Brock Bowers,TE,player,7187.360409,7187.172426,23.0
8,9,Justin Jefferson,WR,player,6614.880056,6713.637183,27.0
9,10,Malik Nabers,WR,player,6532.657105,6698.953617,23.0


### 1QB Dynasty -- 12 team -- Top 200

In [ ]:
boards[("oneqb_dynasty", 12)][
    ["board_rank", "name", "position", "asset_type", "model_value", "consensus_value", "current_age"]
].head(200)

,board_rank,name,position,asset_type,model_value,consensus_value,current_age
0,1,Jahmyr Gibbs,RB,player,11167.526411,11011.192293,24.0
1,2,Bijan Robinson,RB,player,10997.875567,11184.307330,24.0
2,3,Ja'Marr Chase,WR,player,9975.500107,10035.315479,26.0
3,4,Jaxon Smith-Njigba,WR,player,8781.732164,8834.612534,24.0
4,5,Puka Nacua,WR,player,8180.895210,8278.514246,25.0
5,6,Ashton Jeanty,RB,player,7528.985883,7595.214824,22.0
6,7,Amon-Ra St. Brown,WR,player,7324.179070,7232.577143,26.0
7,8,Brock Bowers,TE,player,7115.261807,7194.785882,23.0
8,9,Justin Jefferson,WR,player,6676.597652,6753.072305,27.0
9,10,Malik Nabers,WR,player,6589.121942,6738.272511,23.0


### Redraft -- 10 team -- Top 200

In [ ]:
boards[("redraft", 10)][
    ["board_rank", "name", "position", "asset_type", "model_value", "consensus_value", "current_age"]
].head(200)

,board_rank,name,position,asset_type,model_value,consensus_value,current_age
0,1,Jahmyr Gibbs,RB,player,10414.498872,10200.000000,24.0
1,2,Ja'Marr Chase,WR,player,10007.110349,9757.817630,26.0
2,3,Bijan Robinson,RB,player,9823.374040,10056.487548,24.0
3,4,Puka Nacua,WR,player,9382.529765,9436.893633,25.0
4,5,Jaxon Smith-Njigba,WR,player,9167.186509,9321.934016,24.0
5,6,Amon-Ra St. Brown,WR,player,8818.652624,8650.483990,26.0
6,7,Jonathan Taylor,RB,player,8515.584889,8348.486178,27.0
7,8,Christian McCaffrey,RB,player,8386.366439,8217.410706,30.0
8,9,CeeDee Lamb,WR,player,8014.465345,8080.936695,27.0
9,10,James Cook,RB,player,8013.733156,8149.510939,26.0


### Redraft -- 12 team -- Top 200

In [ ]:
boards[("redraft", 12)][
    ["board_rank", "name", "position", "asset_type", "model_value", "consensus_value", "current_age"]
].head(200)

,board_rank,name,position,asset_type,model_value,consensus_value,current_age
0,1,Jahmyr Gibbs,RB,player,10389.072402,10093.000000,24.0
1,2,Ja'Marr Chase,WR,player,10170.361082,9766.883164,26.0
2,3,Bijan Robinson,RB,player,9729.714654,9951.196029,24.0
3,4,Puka Nacua,WR,player,9281.202798,9492.122868,25.0
4,5,Jaxon Smith-Njigba,WR,player,9147.055616,9376.516174,24.0
5,6,Amon-Ra St. Brown,WR,player,9104.971168,8931.980820,26.0
6,7,Jonathan Taylor,RB,player,8419.735601,8260.942052,27.0
7,8,Christian McCaffrey,RB,player,8094.800763,8012.616734,30.0
8,9,CeeDee Lamb,WR,player,8080.611975,8036.159367,27.0
9,10,James Cook,RB,player,7975.649358,8074.358443,26.0


## Stage 9 -- Divergence tables

Largest model-vs-consensus gaps, player rows only (picks are priced directly from blended market
data, not model-predicted, so they always show zero residual and carry no signal here). These are
**candidates to investigate** -- fundamentals the model weighs differently than the market
currently prices them -- never a claim that the model is right and the market is wrong.

In [ ]:
divergence_tables = tcp.build_all_divergence_tables(boards)
for (fmt, team_size), div_df in divergence_tables.items():
    print(f"\n=== {fmt} - {team_size} team ===")
    display(div_df)

[divergence] excluding 56 rookie row(s) with college_seasons_found==0 from divergence table (no independent signal until CFBD's quota resets)
[divergence] excluding 56 rookie row(s) with college_seasons_found==0 from divergence table (no independent signal until CFBD's quota resets)
[divergence] excluding 56 rookie row(s) with college_seasons_found==0 from divergence table (no independent signal until CFBD's quota resets)
[divergence] excluding 56 rookie row(s) with college_seasons_found==0 from divergence table (no independent signal until CFBD's quota resets)
[divergence] excluding 13 rookie row(s) with college_seasons_found==0 from divergence table (no independent signal until CFBD's quota resets)
[divergence] excluding 13 rookie row(s) with college_seasons_found==0 from divergence table (no independent signal until CFBD's quota resets)

=== sf_dynasty - 10 team ===


,name,position,model_value,consensus_value,residual,residual_pct
0,J.J. McCarthy,QB,1623.450460,1383.251172,240.199288,17.364835
1,Tetairoa McMillan,WR,5334.465910,5103.365342,231.100568,4.528396
2,C.J. Stroud,QB,3375.094151,3198.929494,176.164657,5.506988
3,Joe Mixon,RB,284.375334,109.736842,174.638492,159.142990
4,Justin Fields,QB,836.631805,679.795734,156.836071,23.071058
5,Breece Hall,RB,3985.156447,3839.997196,145.159251,3.780192
6,Harold Fannin,TE,3387.755543,3245.237393,142.518150,4.391609
7,Jahmyr Gibbs,RB,10516.887564,10385.258897,131.628667,1.267457
8,Jalen Hurts,QB,5213.967660,5106.697233,107.270427,2.100583
9,Justin Herbert,QB,5976.223425,5884.489600,91.733825,1.558909



=== sf_dynasty - 12 team ===


,name,position,model_value,consensus_value,residual,residual_pct
0,Joe Mixon,RB,314.626718,109.473684,205.153034,187.399406
1,Baker Mayfield,QB,3488.024410,3304.289531,183.734879,5.560496
2,Tetairoa McMillan,WR,5173.247670,4998.549122,174.698548,3.494985
3,Dak Prescott,QB,4144.961590,3982.149993,162.811597,4.088535
4,J.J. McCarthy,QB,1572.399451,1422.057674,150.341777,10.572129
5,Jahmyr Gibbs,RB,10360.832614,10214.317293,146.515320,1.434411
6,Harold Fannin,TE,3429.315363,3286.540419,142.774945,4.344232
7,C.J. Stroud,QB,3396.659326,3256.916398,139.742928,4.290651
8,Rico Dowdle,RB,1923.873784,1813.576017,110.297767,6.081783
9,Breece Hall,RB,3966.700662,3863.348826,103.351837,2.675188



=== oneqb_dynasty - 10 team ===


...

,name,position,model_value,consensus_value,residual,residual_pct
0,Tyler Warren,TE,4695.643566,4503.793351,191.850215,4.259747
1,Caleb Williams,QB,4000.006690,3808.432319,191.574371,5.030268
2,Christian McCaffrey,RB,4632.736944,4452.226604,180.510341,4.054383
3,Jahmyr Gibbs,RB,11167.526411,11011.192293,156.334118,1.419775
4,Jalen Hurts,QB,3102.653879,2968.160715,134.493164,4.531195
5,Joe Mixon,RB,241.883934,117.383459,124.500475,106.063049
6,J.J. McCarthy,QB,884.961720,768.304902,116.656818,15.183662
7,Zay Flowers,WR,3937.520103,3838.558101,98.962001,2.578104
8,Baker Mayfield,QB,1887.334280,1794.177775,93.156505,5.192156
9,Amon-Ra St. Brown,WR,7324.179070,7232.577143,91.601926,1.266518



=== redraft - 10 team ===


,name,position,model_value,consensus_value,residual,residual_pct
0,Ja'Marr Chase,WR,10007.110349,9757.817630,249.292719,2.554800
1,Travis Etienne,RB,3081.280988,2857.231505,224.049483,7.841489
2,Bo Nix,QB,1673.038792,1456.802028,216.236764,14.843250
3,Jahmyr Gibbs,RB,10414.498872,10200.000000,214.498872,2.102930
4,Brock Bowers,TE,6898.913297,6694.902292,204.011005,3.047259
5,Christian McCaffrey,RB,8386.366439,8217.410706,168.955733,2.056070
6,Amon-Ra St. Brown,WR,8818.652624,8650.483990,168.168634,1.944037
7,Jonathan Taylor,RB,8515.584889,8348.486178,167.098712,2.001545
8,Jared Goff,QB,980.513451,820.597636,159.915815,19.487725
9,Tyler Warren,TE,3118.328103,2984.057130,134.270973,4.499611



=== redraft - 12 team ===


,name,position,model_value,consensus_value,residual,residual_pct
0,Ja'Marr Chase,WR,10170.361082,9766.883164,403.477918,4.131082
1,Jahmyr Gibbs,RB,10389.072402,10093.000000,296.072402,2.933443
2,Bo Nix,QB,1688.359193,1476.987793,211.371400,14.310978
3,Amon-Ra St. Brown,WR,9104.971168,8931.980820,172.990348,1.936752
4,Trey McBride,TE,6869.721531,6698.266194,171.455336,2.559697
5,Jonathan Taylor,RB,8419.735601,8260.942052,158.793549,1.922221
6,Travis Etienne,RB,3028.437605,2873.797663,154.639942,5.381031
7,Trevor Lawrence,QB,2335.886865,2192.795199,143.091666,6.525537
8,D'Andre Swift,RB,2634.743371,2492.527870,142.215501,5.705673
9,Drake Maye,QB,3693.036217,3592.812416,100.223801,2.789564


## Stage 10 -- Sanity panel

~20 fixed, real-world-unambiguous checks (absolute top-N and pairwise orderings), run every
execution, with each player's raw feature values printed alongside any failure so the "why" is
inspectable rather than just a pass/fail flag.

In [ ]:
sanity_results = tcp.run_sanity_panel(boards, feature_table)
sanity_results

!!! SANITY CHECK FAILED !!! Josh Jacobs rank 72 (needed top-60) on ('oneqb_dynasty', 10)
    feature decomposition: position='RB', current_age=np.float64(28.0), career_games=np.float64(105.0), seasons_played=np.float64(7.0), ppg_ppr=np.float64(15.806666666666668), round=np.float64(1.0), pick=np.float64(24.0), dominator_rating=np.float64(nan), speed_score=np.float64(nan)

[sanity_panel] 19 passed, 1 failed, 0 skipped (player not found) of 20 checks


,check,board,passed,detail
0,Puka Nacua top-10,"(sf_dynasty, 12)",True,"rank=7 (need <= 10), model_value=8340"
1,Bijan Robinson top-5,"(sf_dynasty, 12)",True,"rank=2 (need <= 5), model_value=10167"
2,Jahmyr Gibbs top-8,"(sf_dynasty, 12)",True,"rank=1 (need <= 8), model_value=10361"
3,Ja'Marr Chase top-10,"(sf_dynasty, 12)",True,"rank=3 (need <= 10), model_value=10167"
4,Justin Jefferson top-30,"(sf_dynasty, 12)",True,"rank=15 (need <= 30), model_value=6901"
5,Amon-Ra St. Brown top-15,"(sf_dynasty, 12)",True,"rank=9 (need <= 15), model_value=7372"
6,CeeDee Lamb top-20,"(sf_dynasty, 12)",True,"rank=17 (need <= 20), model_value=6472"
7,Josh Allen top-20,"(oneqb_dynasty, 12)",True,"rank=17 (need <= 20), model_value=5668"
8,Lamar Jackson top-20,"(sf_dynasty, 12)",True,"rank=12 (need <= 20), model_value=7068"
9,Brock Bowers top-10,"(sf_dynasty, 12)",True,"rank=8 (need <= 10), model_value=7872"


## Stage 11 -- In-season update

`recompute_board(through_week=None, ...)` reproduces the preseason boards unchanged -- this is the
identity case, confirmed exactly equal below. Once the 2026 season is underway, calling
`tcp.recompute_board(through_week=<int>, feature_table, crosswalk, consensus_targets,
pick_universe, fit_results)` blends that many weeks of real partial-season production into the
feature snapshot (empirical-Bayes shrinkage against the prior-season baseline, with a `MIN_GAMES`
guard against small-sample noise) and re-predicts with the already-fitted models -- no retraining
required. Each call snapshots every board to `snapshots/{date}_{format}_{team_size}.csv` for
week-over-week movement tracking.

In [ ]:
preseason_again = tcp.recompute_board(
    None, feature_table, crosswalk, consensus_targets, pick_universe, fit_results, write_snapshot=False
)
import numpy as np
same = np.allclose(
    preseason_again[("sf_dynasty", 12)]["model_value"].sort_index(),
    boards[("sf_dynasty", 12)]["model_value"].sort_index(),
)
print(f"recompute_board(through_week=None) reproduces the original boards exactly: {same}")
assert same

recompute_board(through_week=None) reproduces the original boards exactly: True


---

## Scope, exclusions, and setup

See `README.md` in this directory for full methodology, known exclusions (K/DST/IDP not modeled,
TE premium not modeled, CFBD rookie college-production features degrade gracefully when the
free-tier monthly quota is exhausted), and setup instructions.

**Setup reminder:** this notebook requires a `CFBD_API_KEY` set in a gitignored `.env` file in this
directory (`CFBD_API_KEY=your_key_here`) -- get a free key at
[collegefootballdata.com/key](https://collegefootballdata.com/key). The key is never hardcoded or
printed anywhere in this notebook or in `trade_calculator_pipeline.py`.